# 07 — ABS Population → `abs_population_by_sa3.csv`

**Source:** `data/raw/abs_population/32350DS0005_2001-24.xlsx` — Table 3 (Persons)  
**Output:** `data/clean/abs_population_by_sa3.csv`  
**Grain:** one row per SA3 × year  
**Key columns:** `sa3_code`, `sa3_name`, `state`, `year`, `total_pop`, `pop_65_plus`

**Logic:**
- Table 3 has absolute counts by 5-year age band at SA2 level, 2001–2024
- `pop_65_plus` = sum of bands: 65–69, 70–74, 75–79, 80–84, 85+
- Aggregate SA2 → SA3 by summing
- Filter to years 2019–2024 (matching service data range)
- `sa3_code` cast to integer string (no trailing `.0`)

In [1]:
import pandas as pd
import numpy as np

FILE = "../../data/raw/abs_population/32350DS0005_2001-24.xlsx"
OUT  = "../../data/clean/abs_population_by_sa3.csv"

YEARS = list(range(2019, 2025))  # 2019–2024 (2025 ABS data not yet released)

STATE_MAP = {
    1: "NSW", 2: "VIC", 3: "QLD", 4: "SA",
    5: "WA",  6: "TAS", 7: "NT",  8: "ACT",
}

## 1. Load raw data

Table 3 has 6 metadata rows before the data (rows 0–5 = description, ABS header, title, publication, blank, age-band labels)  
Row 6 contains geography labels (Year, S/T code, ...) and unit labels (no., no., ...)  
Data starts at row 7 → `skiprows=7`, then assign column names manually.

In [2]:
COLS = [
    "year", "st_code", "st_name", "gccsa_code", "gccsa_name",
    "sa4_code", "sa4_name", "sa3_code", "sa3_name", "sa2_code", "sa2_name",
    "age_0_4", "age_5_9", "age_10_14", "age_15_19", "age_20_24",
    "age_25_29", "age_30_34", "age_35_39", "age_40_44", "age_45_49",
    "age_50_54", "age_55_59", "age_60_64",
    "age_65_69", "age_70_74", "age_75_79", "age_80_84", "age_85plus",
    "total_persons",
]

raw = pd.read_excel(
    FILE,
    sheet_name="Table 3",
    header=None,
    skiprows=7,
    engine="calamine",
)
raw.columns = COLS
print(f"Loaded {len(raw):,} rows")
raw.head(3)

Loaded 58,897 rows


,year,st_code,st_name,gccsa_code,gccsa_name,sa4_code,sa4_name,sa3_code,sa3_name,sa2_code,...,age_45_49,age_50_54,age_55_59,age_60_64,age_65_69,age_70_74,age_75_79,age_80_84,age_85plus,total_persons
0,2001,1.0,New South Wales,1RNSW,Rest of NSW,101.0,Capital Region,10102.0,Queanbeyan,101021007.0,...,221.0,262.0,233.0,197.0,125.0,108.0,79.0,57.0,37.0,2760.0
1,2001,1.0,New South Wales,1RNSW,Rest of NSW,101.0,Capital Region,10102.0,Queanbeyan,101021008.0,...,675.0,585.0,491.0,300.0,223.0,201.0,144.0,64.0,32.0,9129.0
2,2001,1.0,New South Wales,1RNSW,Rest of NSW,101.0,Capital Region,10102.0,Queanbeyan,101021009.0,...,633.0,620.0,520.0,444.0,384.0,394.0,320.0,214.0,226.0,9717.0


## 2. Filter years and compute pop_65_plus

In [3]:
df = raw[raw["year"].isin(YEARS)].copy()
print(f"Rows after year filter ({YEARS[0]}–{YEARS[-1]}): {len(df):,}")

AGE_65_COLS = ["age_65_69", "age_70_74", "age_75_79", "age_80_84", "age_85plus"]
df["pop_65_plus"] = df[AGE_65_COLS].sum(axis=1)

df["state"] = df["st_code"].map(STATE_MAP)

unmapped = df["state"].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows with unmapped state code")
    print(df[df["state"].isna()][["year","st_code","sa3_code","sa3_name"]].drop_duplicates())

Rows after year filter (2019–2024): 14,724
       year  st_code  sa3_code                 sa3_name
46622  2019      9.0   90101.0         Christmas Island
46623  2019      9.0   90102.0  Cocos (Keeling) Islands
46624  2019      9.0   90103.0               Jervis Bay
46625  2019      9.0   90104.0           Norfolk Island
49076  2020      9.0   90101.0         Christmas Island
49077  2020      9.0   90102.0  Cocos (Keeling) Islands
49078  2020      9.0   90103.0               Jervis Bay
49079  2020      9.0   90104.0           Norfolk Island
51530  2021      9.0   90101.0         Christmas Island
51531  2021      9.0   90102.0  Cocos (Keeling) Islands
51532  2021      9.0   90103.0               Jervis Bay
51533  2021      9.0   90104.0           Norfolk Island
53984  2022      9.0   90101.0         Christmas Island
53985  2022      9.0   90102.0  Cocos (Keeling) Islands
53986  2022      9.0   90103.0               Jervis Bay
53987  2022      9.0   90104.0           Norfolk Island
56438

## 3. Aggregate SA2 → SA3

In [4]:
sa3 = (
    df.groupby(["year", "sa3_code", "sa3_name", "state"])[["total_persons", "pop_65_plus"]]
    .sum()
    .reset_index()
    .rename(columns={"total_persons": "total_pop"})
)

print(f"SA3 × year rows: {len(sa3):,}")
print(f"Unique SA3s:      {sa3['sa3_code'].nunique()}")
print(f"Years:            {sorted(sa3['year'].unique())}")
sa3.head()

SA3 × year rows: 2,016
Unique SA3s:      336
Years:            [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


,year,sa3_code,sa3_name,state,total_pop,pop_65_plus
0,2019,10102.0,Queanbeyan,NSW,62840.0,7815.0
1,2019,10103.0,Snowy Mountains,NSW,20273.0,3884.0
2,2019,10104.0,South Coast,NSW,74501.0,21332.0
3,2019,10105.0,Goulburn - Mulwaree,NSW,37725.0,7700.0
4,2019,10106.0,Young - Yass,NSW,37562.0,7437.0


## 4. Fix sa3_code dtype

SA3 code comes as float64 (e.g. `10102.0`) — must cast to integer string `"10102"` to match other files.

In [5]:
print("sa3_code dtype before:", sa3["sa3_code"].dtype)
print("Sample before:", sa3["sa3_code"].head(3).tolist())

sa3["sa3_code"] = (
    pd.to_numeric(sa3["sa3_code"], errors="coerce")
    .dropna()
    .astype(int)
    .astype(str)
)
# Drop any rows where sa3_code became NaN (non-numeric entries)
sa3 = sa3[sa3["sa3_code"].notna()].copy()

print("sa3_code dtype after: ", sa3["sa3_code"].dtype)
print("Sample after:",  sa3["sa3_code"].head(3).tolist())

sa3_code dtype before: float64
Sample before: [10102.0, 10103.0, 10104.0]
sa3_code dtype after:  object
Sample after: ['10102', '10103', '10104']


## 5. Sanity checks

In [6]:
# National totals by year
national = sa3.groupby("year")[["total_pop", "pop_65_plus"]].sum()
national["pct_65_plus"] = national["pop_65_plus"] / national["total_pop"] * 100
print("National totals:")
print(national.round(1))

National totals:
       total_pop  pop_65_plus  pct_65_plus
year                                      
2019  25330050.0    4032335.0         15.9
2020  25644445.0    4186927.0         16.3
2021  25680562.0    4313300.0         16.8
2022  26009479.0    4432367.0         17.0
2023  26647810.0    4559008.0         17.1
2024  27189375.0    4699427.0         17.3


In [7]:
# Check nulls
print("Nulls per column:")
print(sa3[["sa3_code","sa3_name","state","year","total_pop","pop_65_plus"]].isna().sum())

# Check for duplicate SA3 × year
dupes = sa3.duplicated(subset=["sa3_code","year"]).sum()
print(f"\nDuplicate sa3_code × year: {dupes}")

Nulls per column:
sa3_code       0
sa3_name       0
state          0
year           0
total_pop      0
pop_65_plus    0
dtype: int64

Duplicate sa3_code × year: 0


In [8]:
# Top 10 SA3s by pop_65_plus in latest year
latest = sa3[sa3["year"] == sa3["year"].max()]
print(f"Top 10 SA3s by pop_65_plus ({sa3['year'].max()}):")
print(
    latest.nlargest(10, "pop_65_plus")[["sa3_name","state","total_pop","pop_65_plus"]]
    .to_string(index=False)
)

Top 10 SA3s by pop_65_plus (2024):
            sa3_name state  total_pop  pop_65_plus
Mornington Peninsula   VIC   171450.0      48518.0
             Gosford   NSW   181658.0      42508.0
             Geelong   VIC   223939.0      39996.0
            Stirling    WA   229809.0      39088.0
         Onkaparinga    SA   184219.0      37608.0
               Wyong   NSW   173145.0      37172.0
           Fairfield   NSW   198113.0      35541.0
 Whittlesea - Wallan   VIC   281619.0      35118.0
           Dandenong   VIC   204942.0      33736.0
              Monash   VIC   201180.0      33569.0


## 6. Sort and export

In [9]:
out = sa3[["sa3_code","sa3_name","state","year","total_pop","pop_65_plus"]]\
    .sort_values(["sa3_code","year"])\
    .reset_index(drop=True)

out.to_csv(OUT, index=False)
print(f"Exported {len(out):,} rows → {OUT}")
print(f"Shape: {out.shape}")
out.head()

Exported 2,016 rows → ../../data/clean/abs_population_by_sa3.csv
Shape: (2016, 6)


,sa3_code,sa3_name,state,year,total_pop,pop_65_plus
0,10102,Queanbeyan,NSW,2019,62840.0,7815.0
1,10102,Queanbeyan,NSW,2020,63957.0,8186.0
2,10102,Queanbeyan,NSW,2021,64892.0,8500.0
3,10102,Queanbeyan,NSW,2022,65769.0,8723.0
4,10102,Queanbeyan,NSW,2023,66882.0,9020.0
